# VisTAD: Loss Component & Feature Weight Ablation Study (Kaggle GPU)

**Experimental Objectives:**
1. **Experiment 1 (Loss Component Ablation):** Isolates the contribution of Focal Loss, Temporal Dice Loss, and Feature Distillation Loss. Demonstrates why removing any single term triggers optimization instability.
2. **Experiment 2 (Feature Weight Grid Sweep):** Sweeps $w_{\text{feat}} \in [0.0, 0.01, 0.05, 0.10, 0.25, 0.50, 1.0, 2.0]$ to prove that $w_{\text{feat}} = 0.1$ balances the gradient forces $1:1$.
3. **Experiment 3 (Universal Foundation Distillation Ablation):** Evaluates zero-shot cross-domain transfer with vs. without teacher feature distillation ($\lambda_{\text{feat}} = 0.0$ vs. $0.5$).
4. **Grand Artifact Export:** Emits full epoch training trajectories, confusion matrices, and metrics to structured JSON files.


In [ ]:
# --- Step 1: Workspace Setup & Codebase Unpacking ---
import os, sys, shutil, subprocess, base64, io, zipfile

repo_dir = '/kaggle/working/UniFoundation'
os.makedirs(repo_dir, exist_ok=True)
os.chdir(repo_dir)

print('Unpacking self-contained ablation suite codebase...')
b64_bundle = "..."
raw_zip = base64.b64decode(b64_bundle)
with zipfile.ZipFile(io.BytesIO(raw_zip)) as zf:
    zf.extractall('.')
print('Codebase unpacked successfully.')
print(f'Extracted top-level files: {sorted(os.listdir("."))}')


In [ ]:
# --- Step 2: GPU Accelerator Diagnostics ---
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f'Detected {n_gpus} GPU(s):')
    for i in range(n_gpus):
        prop = torch.cuda.get_device_properties(i)
        print(f'  [GPU {i}] {prop.name} | VRAM: {prop.total_memory / 1e9:.2f} GB')


In [ ]:
# --- Step 3: Multi-Dataset Auto-Discovery & Path Linking ---
import os, glob, shutil
os.makedirs('data/swat', exist_ok=True)
os.makedirs('data/smd/test', exist_ok=True)
os.makedirs('data/smd/test_label', exist_ok=True)
os.makedirs('data/gecco', exist_ok=True)
os.makedirs('data/msl', exist_ok=True)
os.makedirs('cache', exist_ok=True)
os.makedirs('results/ablations', exist_ok=True)

print('Searching for datasets and caches in /kaggle/input/ ...')
# Link caches from input datasets
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.pt') and 'cache' in f:
            src_p = os.path.join(root, f)
            dst_p = os.path.join('cache', f)
            if not os.path.exists(dst_p):
                try:
                    os.symlink(src_p, dst_p)
                    print(f'Linked cache: {f}')
                except Exception:
                    shutil.copy2(src_p, dst_p)
                    print(f'Copied cache: {f}')
        elif f == 'gecco_water.csv':
            dst_p = 'data/gecco/gecco_water.csv'
            if not os.path.exists(dst_p):
                shutil.copy2(os.path.join(root, f), dst_p)
        elif 'SWaT_Dataset' in f:
            dst_p = os.path.join('data/swat', f)
            if not os.path.exists(dst_p):
                shutil.copy2(os.path.join(root, f), dst_p)

print(f'Available caches: {sorted(os.listdir("cache"))}')


In [ ]:
# --- Step 4: Execute Loss Component & Weight Ablation Suite ---
import os, subprocess, time
os.environ['PYTHONPATH'] = '.'

cmd = [
    sys.executable, 'run_loss_and_weight_ablation.py',
    '--dataset', 'all',
    '--cache_dir', 'cache',
    '--data_dir', 'data',
    '--output_dir', 'results/ablations',
    '--epochs', '15',
    '--batch_size', '64',
    '--lr', '1e-3'
]

print('Starting Loss and Weight Ablation Execution...')
t0 = time.time()
res = subprocess.run(cmd, capture_output=False)
elapsed = time.time() - t0
print(f'Ablation Suite Finished in {elapsed:.1f}s with exit code {res.returncode}')


In [ ]:
# --- Step 5: Universal Foundation Model Distillation Ablation ---
# Compares zero-shot transfer with teacher feature distillation vs standalone
import os, subprocess, json
os.environ['PYTHONPATH'] = '.'

uni_ablation_results = []
for lambda_f in [0.0, 0.5]:
    run_tag = f'uni_distill_lambda_{lambda_f:.1f}'
    print(f'\n[Universal Foundation] Training LODO Fold 1 with lambda_feat = {lambda_f}...')
    cmd = [
        sys.executable, 'train_universal_foundation.py',
        '--train_datasets', 'smd,gecco,msl',
        '--eval_dataset', 'gecco',
        '--epochs', '5',
        '--batch_size', '32',
        '--distill_feat_weight', str(lambda_f),
        '--distill_score_weight', '0.0',
        '--checkpoint_dir', f'checkpoints/{run_tag}',
        '--output_json', f'results/ablations/{run_tag}.json'
    ]
    subprocess.run(cmd, check=False)
    json_out = f'results/ablations/{run_tag}.json'
    if os.path.exists(json_out):
        with open(json_out) as f:
            data = json.load(f)
            data['lambda_feat'] = lambda_f
            uni_ablation_results.append(data)

uni_summary_path = 'results/ablations/universal_foundation_distill_ablation.json'
with open(uni_summary_path, 'w') as f:
    json.dump(uni_ablation_results, f, indent=2)
print(f'[Universal Foundation] Ablation summary saved to: {uni_summary_path}')


In [ ]:
# --- Step 6: Grand Milestone Checkpoint & Export ---
import os, glob, shutil, json

out_export = '/kaggle/working/results'
os.makedirs(out_export, exist_ok=True)

# Copy all JSON results and Markdown reports to permanent results directory
for f in glob.glob('results/**/*.json', recursive=True):
    dst = os.path.join(out_export, os.path.basename(f))
    shutil.copy2(f, dst)
    print(f'Exported JSON: {os.path.basename(f)}')

for f in glob.glob('results/**/*.md', recursive=True):
    dst = os.path.join(out_export, os.path.basename(f))
    shutil.copy2(f, dst)
    print(f'Exported Markdown: {os.path.basename(f)}')

print('\nGrand Milestone Export Completed. All outputs preserved for download.')
